In [1]:
%pip install --upgrade google-adk google-cloud-aiplatform litellm requests ipdb google-cloud-modelarmor --quiet

In [18]:
import os
from typing import Dict, List, Optional
from IPython.display import display, Markdown
import requests
import vertexai
from google.adk.agents import Agent, SequentialAgent
from google.adk import Workflow
from google.adk.models.lite_llm import LiteLlm
from google.adk.models import LlmResponse, LlmRequest
from google.adk.agents.callback_context import CallbackContext
from vertexai.preview import reasoning_engines
from google.adk.tools import google_search
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("safetydance_agent")

MODEL_GEMINI_FLASH = "gemini-2.5-flash"

# Pull secrets and config from the environment — never hardcode these.
GOOGLE_MAPS_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")  # required by LiteLLM for Claude
PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT")
LOCATION = os.getenv("GOOGLE_CLOUD_LOCATION", "us-central1")

for name, value in [
    ("GOOGLE_MAPS_API_KEY", GOOGLE_MAPS_API_KEY),
    ("ANTHROPIC_API_KEY", ANTHROPIC_API_KEY),
    ("GOOGLE_CLOUD_PROJECT", PROJECT_ID),
]:
    if not value:
        print(f"WARNING: {name} is not set in the environment.")

vertexai.init(project=PROJECT_ID, location=LOCATION)

In [4]:
def get_lat_lon(location: str) -> Optional[Dict[str, float]]:
    """
    Convert a place name or address into latitude and longitude using the
    Google Maps Geocoding API.

    Args:
        location (str): A place name, city, or address (e.g., "College Station, TX").

    Returns:
        Optional[Dict[str, float]]: A dictionary with 'lat' and 'lon' keys.
        Returns None if the location cannot be found or an error occurs.
    """
    # breakpoint()
    if not GOOGLE_MAPS_API_KEY:
        return "google api key not found"

    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": location, "key": GOOGLE_MAPS_API_KEY}

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        if data.get("status") != "OK" or not data.get("results"):
            return None

        coords = data["results"][0]["geometry"]["location"]
        return {"lat": coords["lat"], "lon": coords["lng"]}
    except (requests.RequestException, KeyError, IndexError):
        return "exception, try again"

In [5]:
def get_gov_weather_forecast(
    lat: float, lon: float
) -> Optional[List[Dict[str, str]]]:
    """
    Fetch today's weather forecast from the U.S. National Weather Service
    API based on a given latitude and longitude.

    Args:
        lat (float): Latitude of the location (e.g., 38.8977).
        lon (float): Longitude of the location (e.g., -77.0365).

    Returns:
        Optional[List[Dict[str, str]]]: A list of forecast period dictionaries,
        each with 'name', 'temperature', 'shortForecast', and 'detailedForecast'.
        Returns None if data is unavailable (including for non-US locations,
        which NWS does not cover) or an error occurs.
    """
    # NWS requires a descriptive User-Agent identifying the application.
    headers = {"User-Agent": "(google-lab-readynow-weather-agent)"}

    try:
        points_url = f"https://api.weather.gov/points/{lat},{lon}"
        points_resp = requests.get(points_url, headers=headers, timeout=10)
        points_resp.raise_for_status()
        forecast_url = points_resp.json()["properties"]["forecast"]

        forecast_resp = requests.get(forecast_url, headers=headers, timeout=10)
        forecast_resp.raise_for_status()
        periods = forecast_resp.json()["properties"]["periods"]

        return [
            {
                "name": period["name"],
                "temperature": f"{period['temperature']}°{period['temperatureUnit']}",
                "shortForecast": period["shortForecast"],
                # "detailedForecast": period["detailedForecast"],
            }
            for period in periods
        ]
    except (requests.RequestException, KeyError):
        return None

In [6]:

from google.api_core.client_options import ClientOptions
from google.cloud import modelarmor_v1

_MODEL_ARMOR_LOCATION = os.getenv("MODEL_ARMOR_LOCATION", "us-central1")
_MODEL_ARMOR_TEMPLATE = os.getenv("MODEL_ARMOR_TEMPLATE", "safety-dance")

_model_armor_client = modelarmor_v1.ModelArmorClient(
    client_options=ClientOptions(
        api_endpoint=f"modelarmor.{_MODEL_ARMOR_LOCATION}.rep.googleapis.com"
    )
)
_model_armor_template_name = (
    f"projects/{PROJECT_ID}/locations/{_MODEL_ARMOR_LOCATION}/templates/{_MODEL_ARMOR_TEMPLATE}"
)


def check_user_input(user_input: str) -> str:
    """
    Screen user input for harmful, abusive, or malicious content using
    Google Cloud Model Armor.

    Args:
        user_input (str): The raw text a user submitted to the agent.

    Returns:
        str: "BAD" if Model Armor flags the input, "OK" otherwise. Fails
        open (returns "OK") if the Model Armor call itself errors, so a
        service outage doesn't block legitimate users.
    """
    try:
        response = _model_armor_client.sanitize_user_prompt(
            request=modelarmor_v1.SanitizeUserPromptRequest(
                name=_model_armor_template_name,
                user_prompt_data=modelarmor_v1.DataItem(text=user_input),
            )
        )
        match_state = response.sanitization_result.filter_match_state
        return "BAD" if match_state == modelarmor_v1.FilterMatchState.MATCH_FOUND else "OK"
    except Exception:
        return "OK"

In [7]:
def log_user_prompt(
    callback_context: CallbackContext, llm_request: LlmRequest
) -> Optional[LlmResponse]:
    """
    Log the most recent user message before it is sent to the model.

    Args:
        callback_context (CallbackContext): Context for the current agent
            invocation, including the agent's name.
        llm_request (LlmRequest): The request about to be sent to the model.

    Returns:
        Optional[LlmResponse]: Always returns None, allowing processing
        to continue.
    """
    if llm_request.contents:
        last = llm_request.contents[-1]
        if last.role == "user" and last.parts and last.parts[0].text:
            logger.info(
                "[%s] USER » %s", callback_context.agent_name, last.parts[0].text.strip()
            )

    return None

In [8]:
def log_model_response(
    callback_context: CallbackContext, llm_response: LlmResponse
) -> Optional[LlmResponse]:
    """
    Log the model's response after it is generated, before it is returned
    to the user.

    Args:
        callback_context (CallbackContext): Context for the current agent
            invocation, including the agent's name.
        llm_response (LlmResponse): The response generated by the model.

    Returns:
        Optional[LlmResponse]: Always returns None, allowing processing
        to continue.
    """
    if llm_response.content and llm_response.content.parts:
        text = llm_response.content.parts[0].text
        if text:
            logger.info("[%s] MODEL » %s", callback_context.agent_name, text.strip())

    return None

In [9]:
def moderate_user_prompt(callback_context: CallbackContext,llm_request: LlmRequest
) -> Optional[LlmResponse]:
    try:
        if not llm_request.contents:
            return None

        last = llm_request.contents[-1]
        if last.role != "user" or not last.parts or not last.parts[0].text:
            return None

        user_text = last.parts[0].text.strip()
        result_text = check_user_input(user_text)

        if result_text.strip().upper() == "BAD":
            return LlmResponse(content={
                "role": "model",
                "parts": [{"text": "⚠️ Sorry, that message violates our content guidelines."}]
            })

    except Exception as e:
        import logging
        logging.exception("Moderation callback failed: %s", e)

    return None  # Proceed with model call


In [10]:
def chained_before_callback(callback_context, llm_request):
# 1. Moderation check
  moderation_result = moderate_user_prompt(callback_context, llm_request)
  if  moderation_result is not None:
    return moderation_result  # STOP: message was inappropriate
# 2. Log user input (optional)
  log_user_prompt(callback_context, llm_request)
  return None
# Allow agent to proceed

In [11]:
def search_for_fun(location: str, weather: str) -> str:
  """
  Find fun things to do in a given location based on the current weather.

  Args:
      location (string): College Station, TX
      weather (string): Most days will see temperatures in the high 90s to low 100s, with heat index values reaching as high as 105°F early in the week.

  Returns:
      Optional[List[Dict[str, str]]]: A list of forecast period dictionaries,
      each with 'name', 'temperature', 'shortForecast', and 'detailedForecast'.
      Returns None if data is unavailable (including for non-US locations,
      which NWS does not cover) or an error occurs.
  """
  query = f"fun things to do in {location} when the weather outside is {weather}"
  google_search_result = google_search.run(query)
  print(type(google_search_result))
  return google_search_result

In [19]:
weather_agent_with_moderation = Agent(
   name="marvin",
   model="gemini-2.5-flash",
   instruction="Get today's weather for {location} and return the information to the main agent",
   tools=[get_lat_lon, get_gov_weather_forecast],
   output_key="weather_today",
)
validator = Agent(
    name="ValidateInput",
    model="gemini-2.5-flash",
    instruction="If the user has provided a valid location, respond with only the location name.",
    output_key="location",
)

fun_finder = Agent(
    name="FunFinder",
    model="gemini-2.5-flash",
    instruction="Find fun things to do today based on {weather_today}.",
    output_key="result",
    tools=[google_search],
)

main_agent = SequentialAgent(
    name="MainAgent",
    description="Verify location with the valitador, check weather with weather_agent_with_moderation, then use fun_finder to find fun things to do.",
    sub_agents=[validator, weather_agent_with_moderation, fun_finder],
)

/tmp/ipykernel_103090/3695537243.py:23: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  main_agent = SequentialAgent(


In [20]:
from vertexai.preview import reasoning_engines
app = reasoning_engines.AdkApp(
   agent=main_agent
)
test_user_id = "test-user-id"

you_are_here = input("Where are you today? ")

while you_are_here != "quit":
    session = app.create_session(user_id=test_user_id)
    display(Markdown(f"## Fun things to do in {you_are_here}"))
    for event in app.stream_query(
        user_id=test_user_id,
        session_id=session['id'],
        message=f"{you_are_here}",
    ):
        if 'finish_reason' in event.keys():
            try:
                display(Markdown(event['content']['parts'][0]['text'].replace('\n','  \n')))
            except:
                pass
    you_are_here = input("Where are you today? ")



Where are you today? Boston, MA


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:975: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


## Fun things to do in Boston, MA

Boston, MA

/usr/local/lib/python3.12/dist-packages/google/adk/models/llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


...

## Fun things to do in cardboard box

That is not a valid location.

I cannot find weather for a "cardboard box" as it is not a valid location. Please try again with a valid location.

I understand you're looking for fun things to do! However, "cardboard box" isn't a real place where I can find weather information or local activities.  
  
To help me find the best activities for you, could you please tell me your current location or the location you're interested in (e.g., city, state, or even a specific address)?

Where are you today? quit
